# IMPORTS

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
import networkx as nx
import nltk
from nltk import ngrams
from nltk.tokenize import TweetTokenizer
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split, learning_curve

from imblearn.over_sampling import SMOTE
import joblib

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF
import pyLDAvis 
import pyLDAvis.lda_model 




**NLTK DOWNLOADS**

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')

# LOADING DATASETS

In [ ]:
training = pd.read_csv("twitter_training.csv", header=None)
validation = pd.read_csv("twitter_validation.csv", header=None)

training.columns = ['id', 'topic', 'label', 'text']
validation.columns = ['id', 'topic', 'label', 'text']


# PRE-PROCESSING


In [ ]:
training['text'] = training['text'].fillna('')


In [ ]:
training.duplicated().sum()

**INITIALIZING TOKENIZER AND LEMMATIZER**

In [ ]:
tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=False)
lemmatizer = WordNetLemmatizer()

**PRE-PROCESSING FUNCTION TO PRE-PROCESS BOTH DATASETS**

In [ ]:
url_re = re.compile(r'https?://\S+|www\.\S+')
mention_re = re.compile(r'@([A-Za-z0-9_]+)')
hashtag_re = re.compile(r'#(\w+)')
emoji_re = re.compile(r'[\U00010000-\U0010ffff]')
punct_re = re.compile(r"[^\w\s']")

def preprocess_dataset(df):
    processed_rows = []
    for text in df['text'].astype(str):
        original = text


        #Extracts hashtags and mentions as separate features
        hashtags = re.findall(r'#(\w+)', text)
        mentions = re.findall(r'@([A-Za-z0-9_]+)', text)


        #Removes urls, mentions, hashtags, emojis and special characters
        text_cleaned = url_re.sub('', text)
        text_cleaned = mention_re.sub('', text_cleaned)
        text_cleaned = hashtag_re.sub('', text_cleaned)
        text_cleaned = emoji_re.sub('', text_cleaned)
        text_cleaned = punct_re.sub(' ', text_cleaned)
        text_cleaned = re.sub(r'\s+', ' ', text_cleaned).strip()


        #Tokenization
        tokens = tokenizer.tokenize(text_cleaned)
        #POS and Lemma
        pos_tags = pos_tag(tokens)
        lemmas = []
        for tok, pos in pos_tags:
            if pos.startswith('J'):
                wn_pos = wordnet.ADJ
            elif pos.startswith('V'):
                wn_pos = wordnet.VERB
            elif pos.startswith('N'):
                wn_pos = wordnet.NOUN
            elif pos.startswith('R'):
                wn_pos = wordnet.ADV
            else:
                wn_pos = wordnet.NOUN
            lemma = lemmatizer.lemmatize(tok, wn_pos)
            lemmas.append(lemma)


        clean_text_for_vector = ' '.join(lemmas)

        #Keeping token and lemmas just incase
        processed_rows.append({
        'clean_text': clean_text_for_vector,
        'hashtags': hashtags,
        'mentions': mentions,
        'tokens': tokens,
        'lemmas': lemmas
        })
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(processed_rows)], axis=1)

**PRE-PROCESSING BOTH DATASETS**

In [ ]:
pro_train = preprocess_dataset(training)
pro_val = preprocess_dataset(validation)

**LABEL ENCODING AND VECTORIZATION**

In [ ]:
#Encoding the sentiment column (label)
le = LabelEncoder()

pro_train['label'] = le.fit_transform(pro_train['label'])
pro_val['label'] = le.transform(pro_val['label'])

y_train = pro_train['label']
y_val = pro_val['label']

vectorizer = TfidfVectorizer(
lowercase=True,
analyzer='word',
token_pattern=r"(?u)\b\w+\b",
sublinear_tf=True,#Helps reduce dominance of very frequent words
max_df=0.95, #Ignores very common words
min_df=5,#Ignores rare words
ngram_range=(1,2)#Gets more context from uni and bigrams
)

#fit on training data
train_data = pro_train['clean_text'].astype(str).tolist()
X_train = vectorizer.fit_transform(train_data)


#transform validation data
val_data = pro_val['clean_text'].astype(str).tolist()
X_val = vectorizer.transform(val_data)


print('Training TF-IDF shape:', X_train.shape)
print('Validation TF-IDF shape:', X_val.shape)

In [ ]:
for idx, label in enumerate(le.classes_):
    print(f"{idx}: {label}")

Exploratory Data

In [ ]:
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
def dataset_overview(train_df, val_df):
    print("="*80)
    print("DATASET OVERVIEW")
    print("="*80)
    
    print(f"\nTraining Set: {len(train_df):,} tweets")
    print(f"Validation Set: {len(val_df):,} tweets")
    print(f"Total: {len(train_df) + len(val_df):,} tweets")

    print("\nMissing Values:")
    print("Training:")
    print(train_df.isnull().sum())
    print("\nValidation:")
    print(val_df.isnull().sum())


    print("\n" + "="*80)
    print("LABEL DISTRIBUTION")
    print("="*80)
    print("\nTraining Set:")
    print(train_df['label'].value_counts())
    print(f"\nPercentages:")
    print(train_df['label'].value_counts(normalize=True) * 100)
    
    print("\nValidation Set:")
    print(val_df['label'].value_counts())
    print(f"\nPercentages:")
    print(val_df['label'].value_counts(normalize=True) * 100)

    # Topic distribution
    print("\n" + "="*80)
    print("TOP 10 TOPICS")
    print("="*80)
    print("\nTraining Set:")
    print(train_df['topic'].value_counts().head(10))
    
    print("\nValidation Set:")
    print(val_df['topic'].value_counts().head(10))
    
    # Text length statistics
    train_df['text_length'] = train_df['text'].astype(str).str.len()
    val_df['text_length'] = val_df['text'].astype(str).str.len()
    
    train_df['word_count'] = train_df['tokens'].apply(len)
    val_df['word_count'] = val_df['tokens'].apply(len)
    
    train_df['num_hashtags'] = train_df['hashtags'].apply(len)
    val_df['num_hashtags'] = val_df['hashtags'].apply(len)
    
    train_df['num_mentions'] = train_df['mentions'].apply(len)
    val_df['num_mentions'] = val_df['mentions'].apply(len)
    
    print("\n" + "="*80)
    print("TEXT STATISTICS")
    print("="*80)
    print("\nTraining Set:")
    print(f"  Avg text length: {train_df['text_length'].mean():.1f} chars")
    print(f"  Avg word count: {train_df['word_count'].mean():.1f} words")
    print(f"  Avg hashtags: {train_df['num_hashtags'].mean():.2f}")
    print(f"  Avg mentions: {train_df['num_mentions'].mean():.2f}")
    print(f"  Total hashtags: {train_df['num_hashtags'].sum()}")
    print(f"  Total mentions: {train_df['num_mentions'].sum()}")
    
    print("\nValidation Set:")
    print(f"  Avg text length: {val_df['text_length'].mean():.1f} chars")
    print(f"  Avg word count: {val_df['word_count'].mean():.1f} words")
    print(f"  Avg hashtags: {val_df['num_hashtags'].mean():.2f}")
    print(f"  Avg mentions: {val_df['num_mentions'].mean():.2f}")
    print(f"  Total hashtags: {val_df['num_hashtags'].sum()}")
    print(f"  Total mentions: {val_df['num_mentions'].sum()}")

SENTIMENT DISTRIBUTION VISUALIZATIONS

In [ ]:
def sentiment_distribution(df, title="Dataset"):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    #Count plot
    sentiment_counts = df['label'].value_counts()
    
    #Creates color mapping for any sentiment values
    sentiment_labels = [str(s) for s in sentiment_counts.index]
    default_colors = ['#2ecc71', '#e74c3c', '#95a5a6', '#3498db', '#9b59b6', '#f39c12']
    colors_list = default_colors[:len(sentiment_labels)]
    
    bars = axes[0].bar(sentiment_labels, sentiment_counts.values, color=colors_list)
    axes[0].set_xlabel('Sentiment', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Count', fontsize=12, fontweight='bold')
    axes[0].set_title(f'Sentiment Distribution - {title}', fontsize=14, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    
    #Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height):,}',
                    ha='center', va='bottom', fontweight='bold')
    
    #Makes a Pie chart
    axes[1].pie(sentiment_counts.values, labels=sentiment_labels, 
            autopct='%1.1f%%', colors=colors_list, startangle=90)
    axes[1].set_title(f'Sentiment Proportions - {title}', fontsize=14, fontweight='bold')
    
    #Sentiment by topic
    top_topics = df['topic'].value_counts().head(5).index
    topic_sentiment = df[df['topic'].isin(top_topics)].groupby(['topic', 'label']).size().unstack(fill_value=0)
    topic_sentiment.plot(kind='bar', stacked=True, ax=axes[2], color=colors_list[:len(topic_sentiment.columns)])
    axes[2].set_xlabel('Topic', fontsize=12, fontweight='bold')
    axes[2].set_ylabel('Count', fontsize=12, fontweight='bold')
    axes[2].set_title(f'Sentiment by Top 5 Topics - {title}', fontsize=14, fontweight='bold')
    axes[2].legend(title='Sentiment', bbox_to_anchor=(1.05, 1))
    axes[2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()


TEXT LENGTH ANALYSIS

In [ ]:
def text_length_analysis(df, title="Dataset"):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    sentiments = df['label'].unique()
    default_colors = ['#2ecc71', '#e74c3c', '#95a5a6', '#3498db', '#9b59b6', '#f39c12']
    color_map = {s: default_colors[i % len(default_colors)] for i, s in enumerate(sentiments)}
    
    #Text length distribution
    for sentiment in sentiments:
        data = df[df['label'] == sentiment]['text_length']
        axes[0, 0].hist(data, alpha=0.6, label=str(sentiment), bins=30,
                    color=color_map[sentiment])
    axes[0, 0].set_xlabel('Text Length (characters)', fontsize=11)
    axes[0, 0].set_ylabel('Frequency', fontsize=11)
    axes[0, 0].set_title('Text Length Distribution', fontsize=13, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    #Word count distribution
    for sentiment in sentiments:
        data = df[df['label'] == sentiment]['word_count']
        axes[0, 1].hist(data, alpha=0.6, label=str(sentiment), bins=30,
                    color=color_map[sentiment])
    axes[0, 1].set_xlabel('Word Count', fontsize=11)
    axes[0, 1].set_ylabel('Frequency', fontsize=11)
    axes[0, 1].set_title('Word Count Distribution', fontsize=13, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    
    #Average text length by sentiment
    avg_length = df.groupby('label')['text_length'].mean().sort_values(ascending=False)
    bars = axes[1, 0].bar([str(x) for x in avg_length.index], avg_length.values,
                        color=[color_map[x] for x in avg_length.index])
    axes[1, 0].set_xlabel('Sentiment', fontsize=11)
    axes[1, 0].set_ylabel('Average Text Length', fontsize=11)
    axes[1, 0].set_title('Average Text Length by Sentiment', fontsize=13, fontweight='bold')
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    for bar in bars:
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}', ha='center', va='bottom')
    
    #Average word count by sentiment
    avg_words = df.groupby('label')['word_count'].mean().sort_values(ascending=False)
    bars = axes[1, 1].bar([str(x) for x in avg_words.index], avg_words.values,
                        color=[color_map[x] for x in avg_words.index])
    axes[1, 1].set_xlabel('Sentiment', fontsize=11)
    axes[1, 1].set_ylabel('Average Word Count', fontsize=11)
    axes[1, 1].set_title('Average Word Count by Sentiment', fontsize=13, fontweight='bold')
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    for bar in bars:
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}', ha='center', va='bottom')
    
    plt.suptitle(f'Text Statistics - {title}', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()

WORD FREQUENCY ANALYSIS

In [ ]:
def word_frequency_analysis(df, top_n=20):
    sentiments = df['label'].unique()
    n_sentiments = len(sentiments)
    
    fig, axes = plt.subplots(1, n_sentiments, figsize=(6*n_sentiments, 6))
    if n_sentiments == 1:
        axes = [axes]
    
    default_colors = ['#2ecc71', '#e74c3c', '#95a5a6', '#3498db', '#9b59b6', '#f39c12']
    color_map = {s: default_colors[i % len(default_colors)] for i, s in enumerate(sentiments)}
    
    for idx, sentiment in enumerate(sentiments):
        #Get all lemmas for this sentiment
        all_words = []
        sentiment_df = df[df['label'] == sentiment]
        for lemmas in sentiment_df['lemmas']:
            all_words.extend(lemmas)
        
        #Count frequencies
        word_freq = Counter(all_words).most_common(top_n)
        
        if word_freq:
            words, counts = zip(*word_freq)
            
            axes[idx].barh(range(len(words)), counts, color=color_map[sentiment])
            axes[idx].set_yticks(range(len(words)))
            axes[idx].set_yticklabels(words)
            axes[idx].invert_yaxis()
            axes[idx].set_xlabel('Frequency', fontsize=11, fontweight='bold')
            axes[idx].set_title(f'Top {top_n} Words - Sentiment: {sentiment}', 
                            fontsize=13, fontweight='bold')
            axes[idx].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

N-GRAM ANALYSIS

In [ ]:
def ngram_analysis(df, n=2, top_n=15):
    sentiments = df['label'].unique()
    
    print("\n" + "="*80)
    print(f"{n}-GRAM ANALYSIS")
    print("="*80)
    
    for sentiment in sentiments:
        print(f"\n{'-'*80}")
        #Convert sentiment to string to handle both numeric and text labels
        sentiment_str = str(sentiment).upper() if isinstance(sentiment, str) else str(sentiment)
        print(f"Top {top_n} {n}-grams for sentiment: {sentiment_str}")
        print('-'*80)
        
        #Get all lemmas for this sentiment
        all_lemmas = []
        sentiment_df = df[df['label'] == sentiment]
        for lemmas in sentiment_df['lemmas']:
            all_lemmas.extend(lemmas)
        
        #Generate n-grams
        ngrams_list = list(ngrams(all_lemmas, n))
        ngram_freq = Counter(ngrams_list).most_common(top_n)
        
        for i, (ngram, count) in enumerate(ngram_freq, 1):
            print(f"  {i:2d}. {' '.join(ngram):45s} → {count:5d}")


WORD CLOUDS

In [ ]:
def generate_wordclouds(df):
    sentiments = df['label'].unique()
    n_sentiments = len(sentiments)
    
    fig, axes = plt.subplots(1, n_sentiments, figsize=(7*n_sentiments, 6))
    if n_sentiments == 1:
        axes = [axes]
    
    cmaps = ['Greens', 'Reds', 'Greys', 'Blues', 'Purples', 'Oranges']
    
    for idx, sentiment in enumerate(sentiments):
        #Get all text for this sentiment
        text = ' '.join(df[df['label'] == sentiment]['clean_text'].astype(str))
        
        if text.strip():
            wordcloud = WordCloud(
                width=800, height=400,
                background_color='white',
                colormap=cmaps[idx % len(cmaps)],
                max_words=100,
                relative_scaling=0.5
            ).generate(text)
            
            axes[idx].imshow(wordcloud, interpolation='bilinear')
            axes[idx].axis('off')
            axes[idx].set_title(f'Sentiment: {sentiment}', 
                            fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

HASHTAG ANALYSIS

In [ ]:
def hashtag_analysis(df, top_n=15, title="Dataset"):
    total_hashtags = df['num_hashtags'].sum()
    
    if total_hashtags == 0:
        print("\n" + "="*80)
        print(f"⚠ NO HASHTAGS IN {title.upper()}")
        print("="*80)
        return
    
    print("\n" + "="*80)
    print(f"HASHTAG ANALYSIS - {title.upper()}")
    print("="*80)
    print(f"Total hashtags: {total_hashtags}")
    print(f"Tweets with hashtags: {(df['num_hashtags'] > 0).sum()} / {len(df)} ({(df['num_hashtags'] > 0).mean()*100:.1f}%)")
    
    sentiments = df['label'].unique()
    n_sentiments = len(sentiments)
    
    fig, axes = plt.subplots(1, n_sentiments, figsize=(6*n_sentiments, 6))
    if n_sentiments == 1:
        axes = [axes]
    
    default_colors = ['#2ecc71', '#e74c3c', '#95a5a6', '#3498db', '#9b59b6', '#f39c12']
    color_map = {s: default_colors[i % len(default_colors)] for i, s in enumerate(sentiments)}
    
    for idx, sentiment in enumerate(sentiments):
        #Get all hashtags for this sentiment
        all_hashtags = []
        sentiment_df = df[df['label'] == sentiment]
        for hashtags in sentiment_df['hashtags']:
            all_hashtags.extend(hashtags)
        
        if all_hashtags:
            hashtag_freq = Counter(all_hashtags).most_common(top_n)
            tags, counts = zip(*hashtag_freq)
            
            axes[idx].barh(range(len(tags)), counts, color=color_map[sentiment])
            axes[idx].set_yticks(range(len(tags)))
            axes[idx].set_yticklabels([f'#{tag}' for tag in tags])
            axes[idx].invert_yaxis()
            axes[idx].set_xlabel('Frequency', fontsize=11, fontweight='bold')
            axes[idx].set_title(f'Top {top_n} Hashtags - Sentiment: {sentiment}', 
                            fontsize=13, fontweight='bold')
            axes[idx].grid(axis='x', alpha=0.3)
        else:
            axes[idx].text(0.5, 0.5, f'No hashtags\nfor sentiment {sentiment}', 
                        ha='center', va='center', fontsize=12, 
                        transform=axes[idx].transAxes)
            axes[idx].axis('off')
    
    plt.suptitle(f'Hashtag Distribution - {title}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

MENTION ANALYSIS

In [ ]:
def mention_analysis(df, top_n=20, title="Dataset"):
    all_mentions = []
    for mentions in df['mentions']:
        all_mentions.extend(mentions)
    
    if not all_mentions:
        print("\n" + "="*80)
        print(f"⚠ NO MENTIONS IN {title.upper()}")
        print("="*80)
        return
    
    print("\n" + "="*80)
    print(f"MENTION ANALYSIS - {title.upper()}")
    print("="*80)
    print(f"Total mentions: {len(all_mentions)}")
    print(f"Unique users: {len(set(all_mentions))}")
    print(f"Tweets with mentions: {(df['num_mentions'] > 0).sum()} / {len(df)} ({(df['num_mentions'] > 0).mean()*100:.1f}%)")
    
    mention_counts = Counter(all_mentions).most_common(top_n)
    
    #Create visualization
    fig = plt.figure(figsize=(20, 8))
    gs = fig.add_gridspec(1, 2, width_ratios=[2, 1])
    
    #Network graph
    ax1 = fig.add_subplot(gs[0])
    G = nx.Graph()
    
    for mention, count in mention_counts:
        G.add_node(mention, size=count)
    
    #Add edges for co-mentions
    edge_weights = {}
    for _, row in df.iterrows():
        mentions = row['mentions']
        if len(mentions) > 1:
            for i in range(len(mentions)):
                for j in range(i+1, len(mentions)):
                    if mentions[i] in dict(mention_counts) and mentions[j] in dict(mention_counts):
                        edge = tuple(sorted([mentions[i], mentions[j]]))
                        edge_weights[edge] = edge_weights.get(edge, 0) + 1
    
    for (node1, node2), weight in edge_weights.items():
        if weight > 1:
            G.add_edge(node1, node2, weight=weight)
    
    pos = nx.spring_layout(G, k=3, iterations=50, seed=42)
    node_sizes = [G.nodes[node]['size'] * 30 for node in G.nodes()]
    
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, 
                        node_color='lightblue', alpha=0.7, ax=ax1)
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax1)
    
    if G.edges():
        edge_widths = [G[u][v]['weight'] * 0.5 for u, v in G.edges()]
        nx.draw_networkx_edges(G, pos, alpha=0.3, width=edge_widths, ax=ax1)
    
    ax1.set_title(f'Mention Network - {title}\n(Node size = frequency, Edge width = co-mentions)', 
                fontsize=14, fontweight='bold')
    ax1.axis('off')
    
    #Bar chart
    ax2 = fig.add_subplot(gs[1])
    mentions_list, counts_list = zip(*mention_counts[:20])
    y_pos = range(len(mentions_list))
    ax2.barh(y_pos, counts_list, color='steelblue', alpha=0.7)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels([f'@{m}' for m in mentions_list], fontsize=9)
    ax2.invert_yaxis()
    ax2.set_xlabel('Mention Count', fontsize=11, fontweight='bold')
    ax2.set_title(f'Top 20 Mentioned Users', fontsize=13, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    #Print top mentions by sentiment
    print("\nMentions by Sentiment:")
    for sentiment in df['label'].unique():
        sentiment_df = df[df['label'] == sentiment]
        sentiment_mentions = []
        for mentions in sentiment_df['mentions']:
            sentiment_mentions.extend(mentions)
        
        if sentiment_mentions:
            top_3 = Counter(sentiment_mentions).most_common(3)
            print(f"\n{sentiment}:")
            print(f"  Total: {len(sentiment_mentions)}")
            print(f"  Top 3: {', '.join([f'@{m}({c})' for m, c in top_3])}")


ENGAGEMENT METRICS

In [ ]:
def engagement_metrics(df, title="Dataset"):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    sentiments = df['label'].unique()
    default_colors = ['#2ecc71', '#e74c3c', '#95a5a6', '#3498db', '#9b59b6', '#f39c12']
    color_map = {s: default_colors[i % len(default_colors)] for i, s in enumerate(sentiments)}
    
    #Average hashtags by sentiment
    avg_hashtags = df.groupby('label')['num_hashtags'].mean()
    bars = axes[0, 0].bar([str(x) for x in avg_hashtags.index], avg_hashtags.values,
                        color=[color_map[x] for x in avg_hashtags.index])
    axes[0, 0].set_xlabel('Sentiment', fontsize=11, fontweight='bold')
    axes[0, 0].set_ylabel('Average Hashtags', fontsize=11, fontweight='bold')
    axes[0, 0].set_title('Average Hashtags per Tweet', fontsize=13, fontweight='bold')
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    for bar in bars:
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}', ha='center', va='bottom')
    
    #Average mentions by sentiment
    avg_mentions = df.groupby('label')['num_mentions'].mean()
    bars = axes[0, 1].bar([str(x) for x in avg_mentions.index], avg_mentions.values,
                        color=[color_map[x] for x in avg_mentions.index])
    axes[0, 1].set_xlabel('Sentiment', fontsize=11, fontweight='bold')
    axes[0, 1].set_ylabel('Average Mentions', fontsize=11, fontweight='bold')
    axes[0, 1].set_title('Average Mentions per Tweet', fontsize=13, fontweight='bold')
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    for bar in bars:
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}', ha='center', va='bottom')
    
    #Hashtag distribution (tweets with hashtags)
    for sentiment in sentiments:
        data = df[df['label'] == sentiment]['num_hashtags']
        data_with_hashtags = data[data > 0]
        if len(data_with_hashtags) > 0:
            axes[1, 0].hist(data_with_hashtags, alpha=0.6, label=str(sentiment),
                        bins=range(0, min(int(data_with_hashtags.max())+2, 15)),
                        color=color_map[sentiment])
    axes[1, 0].set_xlabel('Number of Hashtags', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
    axes[1, 0].set_title('Hashtag Count Distribution (Tweets with Hashtags)', 
                        fontsize=13, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    
    #Mention distribution (tweets with mentions)
    for sentiment in sentiments:
        data = df[df['label'] == sentiment]['num_mentions']
        data_with_mentions = data[data > 0]
        if len(data_with_mentions) > 0:
            axes[1, 1].hist(data_with_mentions, alpha=0.6, label=str(sentiment),
                        bins=range(0, min(int(data_with_mentions.max())+2, 15)),
                        color=color_map[sentiment])
    axes[1, 1].set_xlabel('Number of Mentions', fontsize=11, fontweight='bold')
    axes[1, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
    axes[1, 1].set_title('Mention Count Distribution (Tweets with Mentions)', 
                        fontsize=13, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)
    
    plt.suptitle(f'Engagement Metrics - {title}', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()

COMPLETE EDA PIPELINE

In [ ]:
def run_complete_eda(train_df, val_df):
    
    print("\n" + "="*80)
    print("STARTING COMPREHENSIVE EXPLORATORY DATA ANALYSIS")
    print("="*80)
    

    dataset_overview(train_df, val_df)
    

    print("\n→ Generating sentiment distribution visualizations...")
    sentiment_distribution(train_df, "Training Set")
    sentiment_distribution(val_df, "Validation Set")
    

    print("\n→ Analyzing text length distributions...")
    text_length_analysis(train_df, "Training Set")
    text_length_analysis(val_df, "Validation Set")
    

    print("\n→ Analyzing word frequencies...")
    word_frequency_analysis(train_df, top_n=20)
    word_frequency_analysis(val_df, top_n=20)
    

    print("\n→ Performing n-gram analysis...")
    ngram_analysis(train_df, n=2, top_n=15)
    ngram_analysis(val_df, n=2, top_n=15)


    print("\n→ Generating word clouds...")
    generate_wordclouds(train_df)
    generate_wordclouds(val_df)
    

    print("\n→ Analyzing hashtags...")
    hashtag_analysis(train_df, top_n=15, title="Training Set")
    hashtag_analysis(val_df, top_n=15, title="Validation Set")
    

    print("\n→ Analyzing mentions...")
    mention_analysis(train_df, top_n=20, title="Training Set")
    mention_analysis(val_df, top_n=20, title="Validation Set")
    

    print("\n→ Analyzing engagement metrics...")
    engagement_metrics(train_df, "Training Set")
    engagement_metrics(val_df, "Validation Set")
    
    print("\n" + "="*80)
    print("EDA COMPLETE!")
    print("="*80)

EDA Run

In [ ]:
run_complete_eda(pro_train, pro_val)

Supervised ML

Preprocessing functions

In [ ]:
# Initialize tokenizer & lemmatizer
tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=False)
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Remove URLs, mentions, hashtags, special characters
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@([A-Za-z0-9_]+)', '', text)
    text = re.sub(r'#(\w+)', '', text)
    text = re.sub('[\U00010000-\U0010ffff]', '', text)
    text = re.sub(r"[^\w\s']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize & Lemmatize
    tokens = tokenizer.tokenize(text)
    pos_tags = pos_tag(tokens)
    lemmas = []
    for tok, pos in pos_tags:
        if pos.startswith('J'):
            wn_pos = wordnet.ADJ
        elif pos.startswith('V'):
            wn_pos = wordnet.VERB
        elif pos.startswith('N'):
            wn_pos = wordnet.NOUN
        elif pos.startswith('R'):
            wn_pos = wordnet.ADV
        else:
            wn_pos = wordnet.NOUN
        lemmas.append(lemmatizer.lemmatize(tok, wn_pos))
    
    return ' '.join(lemmas)

# Apply preprocessing
training['clean_text'] = training['text'].apply(preprocess_text)
validation['clean_text'] = validation['text'].apply(preprocess_text)

Train & Evaluate Models

In [ ]:
def evaluate_model(y_true, y_pred, model_name="Model"):
    print(f"\n--- {model_name} ---")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred))

#Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_val)
evaluate_model(y_val, lr_pred, "Logistic Regression")



Handle Class Imbalance

In [ ]:
# Check imbalance
class_counts = np.bincount(y_train)
imbalance_ratio = max(class_counts)/min(class_counts)
print("Imbalance ratio:", imbalance_ratio)

if imbalance_ratio > 2:
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    
    # Retrain Logistic Regression with SMOTE
    lr_smote = LogisticRegression(max_iter=1000)
    lr_smote.fit(X_train_res, y_train_res)
    lr_smote_pred = lr_smote.predict(X_val)
    evaluate_model(y_val, lr_smote_pred, "Logistic Regression + SMOTE")


In [ ]:
# Random Forest 
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=20,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_val)

evaluate_model(y_val, rf_pred, "Random Forest ")


In [ ]:
# SVM
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

linear_svc = LinearSVC()
svm_model_fast = CalibratedClassifierCV(linear_svc)  
svm_model_fast.fit(X_train, y_train)
svm_pred_fast = svm_model_fast.predict(X_val)
evaluate_model(y_val, svm_pred_fast, "Fast SVM")


Save Models

In [ ]:
joblib.dump(lr_model, 'logistic_regression_model.pkl')
joblib.dump(rf_model, 'random_forest_model.pkl')
joblib.dump(svm_model_fast, 'svm_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(le, 'label_encoder.pkl')

print("Models saved")


Predict New Tweet

In [ ]:
def predict_sentiment(text, model, vectorizer, label_encoder):
    df = pd.DataFrame({'text': [text]})
    processed = preprocess_dataset(df)
    clean_text = processed.loc[0, 'clean_text']
    X = vectorizer.transform([clean_text])
    pred = model.predict(X)[0]
    sentiment = label_encoder.inverse_transform([pred])[0]
    confidence = model.predict_proba(X).max() if hasattr(model, 'predict_proba') else None
    return sentiment, confidence

tweets = ["I love this product!", "Very bad service, I'm angry."]
for t in tweets:
    sentiment, conf = predict_sentiment(t, lr_model, vectorizer, le)
    conf_str = f" ({conf:.2%})" if conf else ""
    print(f"{t} → {sentiment}{conf_str}")


Unsupervised ML

Apply LDA with 15-20 topics for topic modeling

In [ ]:
_vectorizer = CountVectorizer(max_features=20000)
X_counts = _vectorizer.fit_transform(pro_train['clean_text'])
lda = LatentDirichletAllocation(n_components=15, learning_method='batch', random_state=42)
lda.fit(X_counts)
words = _vectorizer.get_feature_names_out()  
for idx, topic in enumerate(lda.components_):
    print(f"\nTopic #{idx}:")
    top_words_idx = topic.argsort()[-10:][::-1]
    print(" - ".join([words[i] for i in top_words_idx]))

In [ ]:
_vectorizer = CountVectorizer()
X_counts = _vectorizer.fit_transform(pro_train['clean_text'])
lda = LatentDirichletAllocation(n_components=20, learning_method='batch', random_state=42)
lda.fit(X_counts)
words = _vectorizer.get_feature_names_out()  
for idx, topic in enumerate(lda.components_):
    print(f"\nTopic #{idx}:")
    top_words_idx = topic.argsort()[-10:][::-1] 
    print(" - ".join([words[i] for i in top_words_idx]))

Use KMeans clustering on TF-IDF vectors (k=10)


In [ ]:
kmeans = KMeans(n_clusters=10, random_state=42)
kmeans.fit(X_train)

In [ ]:
pro_train['cluster'] = kmeans.labels_
words = vectorizer.get_feature_names_out()
for i in range(10):
    cluster_center = kmeans.cluster_centers_[i]
    top_idx = cluster_center.argsort()[-10:][::-1] 
    print(f"\nCluster {i} top words:")
    print(" - ".join(words[j] for j in top_idx))

Implement NMF for additive topic decomposition

In [ ]:
_vectorizer = CountVectorizer()
X_counts = _vectorizer.fit_transform(pro_train['clean_text'])
nmf_model = NMF(n_components=15, random_state=42)
nmf_model.fit(X_counts)
words = _vectorizer.get_feature_names_out()  
for idx, topic in enumerate(nmf_model.components_):
    print(f"\nTopic #{idx}:")
    top_words_idx = topic.argsort()[-10:][::-1] 
    print(" - ".join([words[i] for i in top_words_idx]))

In [ ]:
pyLDAvis.enable_notebook()  
lda_viz = pyLDAvis.lda_model.prepare(lda, X_counts, _vectorizer)
pyLDAvis.display(lda_viz)

In [ ]:
joblib.dump(lda, 'lda_model.pkl')
joblib.dump(nmf_model, 'nmf_model_model.pkl')
joblib.dump(KMeans, 'KMeans_model.pkl')
print("Models saved")


• Validate topics with human-in-the-loop quality assessment
